In [16]:
import requests
import re
import logging
from typing import Optional, Tuple
from google.adk.agents import LlmAgent
from google.adk.agents.callback_context import CallbackContext
from google.adk.models import LlmRequest, LlmResponse
from types import SimpleNamespace

# -- Logging --
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s %(levelname)s %(name)s %(message)s",
    force=True,
)

logger = logging.getLogger("weather_agent")
logger.setLevel(logging.INFO)
if not logger.handlers:
    logging.basicConfig(level=logging.INFO)

# -- Weather retrieval from NWS API for a given latitude and longitude --
def get_nws_weather(latitude: float, longitude: float) -> dict:

  base_url = "https://api.weather.gov"
  user_agent = "MyWeatherApp (tmontgomery)"
  headers = {"User-Agent": user_agent,"Accept": "application/json"}
  weather_data = {}

  try:
    # -- Get gridpoint metadata --
    points_url = f"{base_url}/points/{latitude},{longitude}"
    points_response = requests.get(points_url, headers=headers)
    points_response.raise_for_status()
    points_data = points_response.json()

    # -- Get city name and state --
    city = points_data.get('properties', {}).get('relativeLocation', {}).get('properties', {}).get('city')
    state = points_data.get('properties', {}).get('relativeLocation', {}).get('properties', {}).get('state')

    # -- Store basic location info --
    weather_data["coordinates"] = {"Latitude": latitude, "Longitude": longitude}
    weather_data["city"] = city if city else "Unknown City"
    weather_data["state"] = state if state else "Unknown State"

    forecast_url = points_data.get('properties', {}).get('forecast')

    # -- Get the detailed forecast --
    forecast_response = requests.get(forecast_url, headers=headers)
    forecast_response.raise_for_status()
    forecast_data = forecast_response.json()

    # -- Extract first forecast period --
    periods = forecast_data.get('properties', {}).get('periods', [])
    current_period = periods[0]
    temperature = f"{current_period.get('temperature')} {current_period.get('temperatureUnit')}"
    detailed_forecast = current_period.get('detailedForecast')

    weather_data["temperature"] = temperature if temperature else "N/A"
    weather_data["detailed_forecast"] = detailed_forecast if detailed_forecast else "No detailed forecast available."

    return weather_data

  except requests.exceptions.RequestException as e:
    print(f"Error fetching weather data: {e}")
    return {}

  except KeyError as e:
    print(f"Error parsing NWS API response: Missing key {e}")
    return {}

  return weather_data

# -- Callbacks --
_ALLOWED_CHARS = re.compile(r"^[0-9\s\-\+\,\.\(\):=a-zA-Z]*$")
_FLOAT = r"[-+]?\d+(?:\.\d+)?"
_LAT_LON_2FLOATS = re.compile(rf"({_FLOAT})\s*[, ]\s*({_FLOAT})")
_LAT_EQ = re.compile(rf"\blat(?:itude)?\s*[:=]\s*({_FLOAT})\b", re.IGNORECASE)
_LON_EQ = re.compile(rf"\b(lon|lng|longitude)\s*[:=]\s*({_FLOAT})\b", re.IGNORECASE)

def _extract_last_user_text(llm_request) -> Optional[str]:
    if not getattr(llm_request, "contents", None):
        return None
    last = llm_request.contents[-1]
    if getattr(last, "role", None) != "user":
        return None
    parts = getattr(last, "parts", None) or []
    if not parts or not getattr(parts[0], "text", None):
        return None
    return parts[0].text.strip()

def _parse_lat_lon(text: str) -> Tuple[float, float]:
    mlat = _LAT_EQ.search(text)
    mlon = _LON_EQ.search(text)
    if mlat and mlon:
        return float(mlat.group(1)), float(mlon.group(2))

    m = _LAT_LON_2FLOATS.search(text)
    if m:
        return float(m.group(1)), float(m.group(2))

    raise ValueError("Could not find latitude/longitude.")

def _is_in_us_approx(lat: float, lon: float) -> bool:
    in_conus = 24.396308 <= lat <= 49.384358 and -124.848974 <= lon <= -66.885444
    in_ak    = 51.2      <= lat <= 71.5      and -179.15     <= lon <= -129.99
    in_hi    = 18.9      <= lat <= 22.3      and -160.3      <= lon <= -154.8
    return in_conus or in_ak or in_hi

def _reject(message: str):
    return LlmResponse(content={"role": "model", "parts": [{"text": message}]})

def log_user_prompt(callback_context, llm_request):
    user_text = _extract_last_user_text(llm_request)
    if user_text:
        logger.info("[%s] USER » %s", callback_context.agent_name, user_text)
    return None

def log_model_response(callback_context, llm_response):
    content = getattr(llm_response, "content", None)
    parts = getattr(content, "parts", None) if content else None
    if parts and getattr(parts[0], "text", None):
        logger.info("[%s] MODEL » %s", callback_context.agent_name, parts[0].text.strip())
    return None

def moderate_user_prompt(callback_context, llm_request):
    try:
        user_text = _extract_last_user_text(llm_request)
        if not user_text:
            return None

        if len(user_text) > 200:
            return _reject("Provide coordinates like: 34.6272, -78.6107.")
        if not _ALLOWED_CHARS.fullmatch(user_text):
            return _reject("Only latitude/longitude input is allowed.")

        lat, lon = _parse_lat_lon(user_text)
        if not (-90 <= lat <= 90 and -180 <= lon <= 180):
            return _reject("Latitude/longitude out of range.")
        if not _is_in_us_approx(lat, lon):
            return _reject("Location must be within the US.")

        return None
    except Exception as e:
        logger.exception("Moderation callback failed: %s", e)
        return _reject("Unable to process coordinates. Use: 34.6272, -78.6107.")

def chained_before_callback(callback_context, llm_request):
    moderation_result = moderate_user_prompt(callback_context, llm_request)
    if moderation_result is not None:
        return moderation_result
    log_user_prompt(callback_context, llm_request)
    return None

# ---- Agent config ----
WEATHER_AGENT_INSTRUCTIONS = (
    "You are a friendly weather agent. "
    "Ask for US latitude/longitude if missing. "
    "Use the weather tool to answer concisely."
)
try:
  weather_agent_with_moderation = LlmAgent(
    name="David",
    model="gemini-2.0-flash",
    description="David the Friendly Weather Agent.",
    instruction=WEATHER_AGENT_INSTRUCTIONS,
    tools=[get_extended_weather_forecast],
    before_model_callback=chained_before_callback,
    after_model_callback=log_model_response,
)
except NameError:
  pass

# -- Test logging --
def test_callback_logging(user_text: str) -> None:
    callback_context = SimpleNamespace(agent_name="David")
    llm_request = SimpleNamespace(contents=[
        SimpleNamespace(role="user", parts=[SimpleNamespace(text=user_text)])
    ])

    result = chained_before_callback(callback_context, llm_request)
    print("before_model_callback returned:", result)

    # Simulate a model response to test after_model_callback logging
    llm_response = SimpleNamespace(content=SimpleNamespace(parts=[SimpleNamespace(text="Test model response")]))
    log_model_response(callback_context, llm_response)

if __name__ == "__main__":
    test_callback_logging("34.6272, -78.6107")      # should log USER + MODEL
    test_callback_logging("34.6272, -178.6107")     # should block (US-only)

# -- Retrieve weather info for lat/long --
if __name__ == "__main__":
    # Example: Elizabethtown, NC
    lat = 34.6272
    lon = -78.6107

    weather_info = get_nws_weather(lat, lon)

    if weather_info:
        print("\n--- Weather Information ---")
        print(f"Coordinates: {weather_info.get('coordinates')}")
        print(f"City: {weather_info.get('city')}, {weather_info.get('state')}")
        print(f"Temperature: {weather_info.get('temperature')}")
        print(f"Detailed Forecast: {weather_info.get('detailed_forecast')}")
    else:
        print("\nFailed to retrieve weather information.")

2026-03-03 17:20:50,626 INFO weather_agent [David] USER » 34.6272, -78.6107
2026-03-03 17:20:50,627 INFO weather_agent [David] MODEL » Test model response
2026-03-03 17:20:50,628 INFO weather_agent [David] MODEL » Test model response


before_model_callback returned: None
before_model_callback returned: model_version=None content=Content(
  parts=[
    Part(
      text='Location must be within the US.'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=None error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=None live_session_resumption_update=None input_transcription=None output_transcription=None avg_logprobs=None logprobs_result=None cache_metadata=None citation_metadata=None interaction_id=None

--- Weather Information ---
Coordinates: {'Latitude': 34.6272, 'Longitude': -78.6107}
City: Elizabethtown, NC
Temperature: 68 F
Detailed Forecast: Mostly sunny, with a high near 68. East wind around 2 mph.
